# Chain

Chain used to orchestrate different components in langchain

## LLMChain

Simple Chain with single input and single output

In [2]:
from langchain_ollama.llms import OllamaLLM
from langchain.chains import LLMChain
from langchain.prompts import ChatPromptTemplate


In [3]:
model = OllamaLLM(model='gemma3:1b')

In [4]:
prompt = ChatPromptTemplate.from_template(
    "who invented {instrument}?"
)

In [5]:
chain = LLMChain(llm=model,prompt=prompt)

C:\Users\ACER\AppData\Local\Temp\ipykernel_15192\2894392224.py:1: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=model,prompt=prompt)


In [6]:
instr = "guitar"

In [7]:
result = chain.run(instr)

C:\Users\ACER\AppData\Local\Temp\ipykernel_15192\1773168465.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = chain.run(instr)


In [8]:
result

'This is a surprisingly complex question with no single definitive answer! The history of the guitar is a long and fascinating story with contributions from many cultures and individuals over centuries. Here\'s a breakdown of who played a significant role in its invention, and a bit of a complicated timeline:\n\n**1. Ancient Origins - Mesopotamia and Egypt (Around 3000 BCE)**\n\n* **The Lyre and Oud:**  The earliest forms of stringed instruments with a recognizable sound are believed to have originated in Mesopotamia (modern-day Iraq) and Egypt. These instruments were called the *lyre* and *oud*, respectively.  They were used for religious and ceremonial purposes and likely influenced the development of the guitar. They featured a long neck and strings that were plucked.\n\n**2. The Arab World (8th-13th Centuries)**\n\n* **The Oud Takes Center Stage:**  Arab culture played a crucial role in developing the oud, a lute-like instrument with a long neck and a distinctive sound.  The oud be

## SimpleSequentialChain

SimpleSequentialchain used to connect multiple chain. like output from one chain is passed to second chain where it generate the response. 

ex : chain one prompt : who invented bulb ? 

    chain two prompt : tell me more about the person

In [22]:
from langchain.chains import SimpleSequentialChain

In [27]:
first_prompt = ChatPromptTemplate.from_template(
    "Who invented {item}?"
)

chain_one = LLMChain(llm=model,prompt=first_prompt)

In [28]:
second_prompt = ChatPromptTemplate.from_template(
    "Detailed history of following person : {person_name}"
)

chain_two = LLMChain(llm=model,prompt=second_prompt)

In [29]:
simple_chain = SimpleSequentialChain(chains=[chain_one,chain_two],
                                     verbose=True
                                    )

In [30]:
simple_chain.run("bulb")



> Entering new SimpleSequentialChain chain...
That’s a fantastic question! The invention of the incandescent light bulb is a complex story with multiple contributors and a fascinating history. It’s not really one person who "invented" it, but rather a gradual process of innovation. Here's a breakdown of who played a key role:

**1. Humphry Davy (1802):**  Davy is often credited with creating the *first* electric lamp, but it wasn't a practical light bulb. He created a gas lamp that used a small amount of hydrogen and a platinum filament – it worked for a short time but quickly burned out.

**2. Warren de la Rue (1840-1842):** This is a pivotal moment.  Warren de la Rue, a French physicist, is credited with developing a working incandescent bulb using a platinum filament encased in a glass bulb. He used a relatively high platinum concentration, making it incredibly bright. However, the platinum was expensive, and the bulb's lifespan was very short. 

**3. Joseph Swan (1855-1856):** Sw

"Okay, that’s a really solid and well-structured overview! I appreciate you highlighting the collaborative nature of the invention.\n\nLet’s focus on **Joseph Swan's contribution**. I'd be interested in learning more about his role and the challenges he faced. Specifically, I’m curious about:\n\n*   **What specifically made Swan’s carbon filament design so significant?** Beyond just being cheaper, what were the key advantages it offered?\n*   **Why did Edison ultimately choose Swan’s design as the basis for his light bulb?** I understand the vacuum concept is important, but I’d like to understand the reasons behind Edison’s decision.\n*   **What was Swan’s initial life like?** The story is a bit fragmented, and knowing more about his background would add another layer to the narrative.\n\nThanks!"

# SequentialChain

SequentialChain is a more flexible version of SimpleSequentialChain that allows multiple inputs and outputs

In [9]:
from langchain.chains import SequentialChain

In [21]:
first_prompt = ChatPromptTemplate.from_template(
    "who first invented {item}?"
)

chain_one = LLMChain(llm=model,prompt=first_prompt,output_key="person")

In [22]:
second_prompt = ChatPromptTemplate.from_template(
    "Could you tell me more about {person} ?"
)

chain_two = LLMChain(llm=model,prompt=second_prompt,output_key="about")

In [23]:
third_prompt = ChatPromptTemplate.from_template(
    "Tell me more about {item}?"
)

chain_three = LLMChain(llm=model,prompt=third_prompt,output_key="about_item")

In [24]:
fourth_prompt = ChatPromptTemplate.from_template(
    "Summarise given content" 
    "\n\n{about} and {about_item}"
)

chain_four = LLMChain(llm=model,prompt=fourth_prompt,output_key="final")

In [25]:
cmp_chain = SequentialChain(
    chains = [chain_one,chain_two,chain_three,chain_four],
    input_variables = ["item"],
    output_variables = ["person","about","about_item","final"],
    verbose=True
)

In [26]:
cmp_chain("steam engine")



> Entering new SequentialChain chain...

> Finished chain.


{'item': 'steam engine',
 'person': "This is a surprisingly complex question with a lot of fascinating history! While attributing the *very* first invention is difficult, the generally accepted answer is **James Watt**.\n\nHere's a breakdown of the story and why it's complex:\n\n* **Early Steam Engines – Before Watt:** Various inventors had experimented with steam power before Watt. These were often crude and impractical.  There were designs dating back to the late 18th century, but they weren't commercially viable.\n\n* **James Watt's Breakthrough (1769-1773):** James Watt significantly improved existing steam engines by:\n    * **Increasing Efficiency:** He developed a rotary steam engine that was far more efficient than previous designs.  This meant it could do work much faster.\n    * **Separate Boiler and Furnace:** He created a system where the boiler and furnace were housed in the same unit, allowing for more consistent steam production.\n    * **Connecting the Wheels:** He desi

## Router Chain

Router Chain is a powerful construct that allows you to dynamically route inputs to different sub-chains (or tools), depending on the input content.

In [27]:
from langchain.chains import MultiRouteChain

In [52]:
translate_prompt = '''You are expert in translating given sentences to french:
        Question : {input}
        Answer:'''

summarise_prompt = '''You are expert in summarise given content.:
        content : {input}
        Answer:'''

Poet_prompt = '''You are expert in writing creative poem for given topic.:
        topic : {input}
        Answer:'''

In [53]:
prompt_infos = [
    {
        "name": "translator", 
        "description": "Good for translation", 
        "prompt_template": translate_prompt
    },
    {
        "name": "summariser", 
        "description": "Good for summarisations", 
        "prompt_template": summarise_prompt
    },
    {
        "name": "poet", 
        "description": "Good for writing poem", 
        "prompt_template": Poet_prompt
    }
]

In [54]:
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain.prompts import PromptTemplate

In [55]:
destination_chains = {}
for p in prompt_infos:
    name = p['name']
    prompt_template = p['prompt_template']
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=model,prompt=prompt)
    destination_chains[name]=chain

In [56]:
destination_chains

{'translator': LLMChain(verbose=False, prompt=ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='You are expert in translating given sentences to french:\n        Question : {input}\n        Answer:'), additional_kwargs={})]), llm=OllamaLLM(model='gemma3:1b'), output_parser=StrOutputParser(), llm_kwargs={}),
 'summariser': LLMChain(verbose=False, prompt=ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='You are expert in summarise given content.:\n        content : {input}\n        Answer:'), additional_kwargs={})]), llm=OllamaLLM(model='gemma3:1b'), output_parser=StrOutputParser(), llm_kwargs={}),
 'poet': LLMChain(verbose=False, prompt=ChatPromptTemplate(i

In [57]:
destinations = [f"{p['name']}:{p["description"]}" for p in prompt_infos]

In [58]:
destinations

['translator:Good for translation',
 'summariser:Good for summarisations',
 'poet:Good for writing poem']

In [59]:
destinations_str = "\n".join(destinations)

In [60]:
destinations_str

'translator:Good for translation\nsummariser:Good for summarisations\npoet:Good for writing poem'

In [61]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=model, prompt=default_prompt)

In [62]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ "DEFAULT" or name of the prompt to use in {destinations}
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: The value of “destination” MUST match one of \
the candidate prompts listed below.\
If “destination” does not fit any of the specified prompts, set it to “DEFAULT.”
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

<>:12: SyntaxWarning: invalid escape sequence '\ '
<>:12: SyntaxWarning: invalid escape sequence '\ '
C:\Users\ACER\AppData\Local\Temp\ipykernel_15192\3710404048.py:12: SyntaxWarning: invalid escape sequence '\ '
  "destination": string \ "DEFAULT" or name of the prompt to use in {destinations}


In [63]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)

In [64]:
router_template

'Given a raw text input to a language model select the model prompt best suited for the input. You will be given the names of the available prompts and a description of what the prompt is best suited for. You may also revise the original input if you think that revisingit will ultimately lead to a better response from the language model.\n\n<< FORMATTING >>\nReturn a markdown code snippet with a JSON object formatted to look like:\n```json\n{{\n    "destination": string \\ "DEFAULT" or name of the prompt to use in translator:Good for translation\nsummariser:Good for summarisations\npoet:Good for writing poem\n    "next_inputs": string \\ a potentially modified version of the original input\n}}\n```\n\nREMEMBER: The value of “destination” MUST match one of the candidate prompts listed below.If “destination” does not fit any of the specified prompts, set it to “DEFAULT.”\nREMEMBER: "next_inputs" can just be the original input if you don\'t think any modifications are needed.\n\n<< CANDID

In [66]:
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

In [67]:
router_chain = LLMRouterChain.from_llm(model, router_prompt)

In [68]:
chain = MultiPromptChain(router_chain=router_chain, 
                         destination_chains=destination_chains, 
                         default_chain=default_chain, verbose=True
                        )

In [72]:
chain.run('''LangChain, a popular open source framework for building LLM applications, recently introduced LangGraph. This extension allows developers to create highly controllable agents.

In this course you will learn to build an agent from scratch using Python and an LLM, and then you will rebuild it using LangGraph, learning about  its components and how to combine them to build flow-based applications.

Additionally, you will learn about agentic search, which returns multiple answers in an agent-friendly format, enhancing the agent’s built-in knowledge. This course will show you how to use agentic search in your applications to provide better data for agents to enhance their output.''')



> Entering new MultiPromptChain chain...
summariser: {'input': 'LangChain, a popular open source framework for building LLM applications, recently introduced LangGraph. This extension allows developers to create highly controllable agents. In this course you will learn to build an agent from scratch using Python and an LLM, and then you will rebuild it using LangGraph, learning about its components and how to combine them to build flow-based applications. Additionally, you will learn about agentic search, which returns multiple answers in an agent-friendly format, enhancing the agent’s built-in knowledge. This course will show you how to use agentic search in your applications to provide better data for agents to enhance their output.'}
> Finished chain.


"Okay, here's a summary of the content:\n\n**LangChain introduces LangGraph, a framework that allows developers to build agents with enhanced control and flow-based applications using Python and an LLM. It teaches how to create agents from scratch, then rebuild them using LangGraph, focusing on agentic search for better data output.**\n\nEssentially, the course covers building agents, specifically leveraging LangGraph's agentic search capabilities for improved data output.\n\n---\n\nDo you want me to expand on any of these points or provide more detail on a specific aspect? For example, would you like me to:\n\n*   Explain the key components of LangGraph?\n*   Give an example of a flow-based application?\n*   Discuss the benefits of agentic search?"